# EXP-010 — Региональные динуклеотиды в окне 201 bp

**Champion:** `EXP-009/run_001` — EXP-007 + fixed CpG O/E201 bins

**Гипотеза:** Другие соседние пары букв содержат дополнительный сигнал о старте транскрипции сверх локальных k-mer, GC201 и CpG O/E201

**Единственное основное изменение:** Поочередно добавить к EXP-009 один O/E-признак с фиксированными bins: AA, TT, TA, GG, CC, AT, GC, CA, TG, AG, CT, GA, TC, AC, GT. Окно [-100,+100], C=0.0001, одинаковые 3 folds: всего 45 fits. Признаки не накапливать

**Критерий успеха:** Train-CV: относительный прирост mean AP не менее 1%, AP выше reference минимум на 2 из 3 folds, mean Spearman не ниже, все fits сошлись. Из прошедших выбрать одного кандидата по mean AP. Validation: относительный прирост AP не менее 1%, Spearman не ниже EXP-009, AP выше минимум на 2 из 3 contig

Notebook создан пустым и ничего не запускает автоматически.


## Карта эксперимента

1. Проверить champion и split. 2. Вместе реализовать новый признак.
3. Сделать дешёвый pilot кардинальности и памяти. 4. Полный train count.
5. Paired CV. 6. Full-train fit. 7. Одноразовый validation. 8. Сохранение.


In [1]:
# Выполнить первой после restart kernel.
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")


'2'

In [2]:
from pathlib import Path
import json
import sys
import time
from IPython.display import display
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (CURRENT_DIR, *CURRENT_DIR.parents)
                    if (p / "README.md").is_file() and (p / "src/ml_project").is_dir())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_data import ensure_prepared_split
from ml_project.epic_experiment import ExperimentSpec, build_run_record, save_run_record


## 1. Зафиксированный протокол


In [76]:
RUN_NAME = "run_001"
EXPERIMENT_ID = 'EXP-010'
CHAMPION_ID = 'EXP-009'
CHAMPION_RUN = 'run_001'
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks/experiments" / 'EXP-010_regionalnye_dinukleotidy_v_okne_201_bp.ipynb'
ARTIFACT_DIR = PROJECT_ROOT / "artifacts/experiments" / EXPERIMENT_ID / RUN_NAME
CHAMPION_DIR = PROJECT_ROOT / "artifacts/experiments" / CHAMPION_ID / CHAMPION_RUN
SPEC = ExperimentSpec(EXPERIMENT_ID, 'Региональные динуклеотиды в окне 201 bp', 'Другие соседние пары букв содержат дополнительный сигнал о старте транскрипции сверх локальных k-mer, GC201 и CpG O/E201',
                      'Поочередно добавить к EXP-009 один O/E-признак с фиксированными bins: AA, TT, TA, GG, CC, AT, GC, CA, TG, AG, CT, GA, TC, AC, GT. Окно [-100,+100], C=0.0001, одинаковые 3 folds: всего 45 fits. Признаки не накапливать', 'Train-CV: относительный прирост mean AP не менее 1%, AP выше reference минимум на 2 из 3 folds, mean Spearman не ниже, все fits сошлись. Из прошедших выбрать одного кандидата по mean AP. Validation: относительный прирост AP не менее 1%, Spearman не ниже EXP-009, AP выше минимум на 2 из 3 contig',
                      primary_metric="average_precision",
                      secondary_metric="epic_spearman", seed=42)
PAIR_TO_RUN = "GT"  # Меняем по очереди согласно DINUCLEOTIDE_PAIRS.
CANDIDATE_PARAMETERS = {"parent": f"{CHAMPION_ID}/{CHAMPION_RUN}",
                        "implementation": 'module',
                        "pair": PAIR_TO_RUN}
assert not (ARTIFACT_DIR / "metadata.json").exists()
display(pd.Series(CANDIDATE_PARAMETERS, name="value").to_frame())


,value
parent,EXP-009/run_001
implementation,module
pair,GT


In [4]:
champion_metadata = json.loads(
    (CHAMPION_DIR / "metadata.json").read_text(encoding="utf-8"))
champion_metrics = pd.read_csv(CHAMPION_DIR / "metrics.csv")
fold_assignment = pd.read_csv(CHAMPION_DIR / "cv_fold_assignment.csv")
assert champion_metadata["run"]["decision"] == "adopt"
assert champion_metadata["experiment"]["experiment_id"] == CHAMPION_ID
display(champion_metrics)


,metric,value
0,reference_average_precision,0.002054
1,candidate_average_precision,0.002364
2,delta_average_precision,0.000310
3,reference_epic_spearman,0.104302
4,candidate_epic_spearman,0.115517
5,delta_epic_spearman,0.011215
6,historical_exp003_average_precision,0.001601
7,historical_exp003_epic_spearman,0.097186
8,selected_cv_candidate_mean_average_precision,0.002672
9,selected_cv_reference_mean_average_precision,0.002324


## 2. Неизменные split и последовательности


In [5]:
split = ensure_prepared_split(PROJECT_ROOT)
display(split.summary)
display(split.contig_assignment.sort_values(["split", "contig"]))
assert split.config.split_version == "contig_holdout_v1" and split.config.seed == SPEC.seed


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


,contig,split,source_split
12,NC_064037.1,test,test
13,NC_064044.1,test,test
14,NC_064046.1,test,test
1,NC_064035.1,train,train
2,NC_064036.1,train,train
3,NC_064038.1,train,train
4,NC_064039.1,train,train
5,NC_064040.1,train,train
8,NC_064043.1,train,train
9,NC_064045.1,train,train


In [6]:
sequences, fasta_alphabet = load_baseline_sequences(split)
display(fasta_alphabet)
print("Loaded contigs:", len(sequences))


{'A': 54173800,
 'C': 39020211,
 'G': 38944862,
 'N': 34600,
 'T': 54153761,
 'a': 25666946,
 'c': 15841270,
 'g': 15877204,
 'n': 600,
 't': 25705184}

Loaded contigs: 47


## 3. Единственное нововведение

Код размещается в **module**. До полного подсчёта фиксируем формулу,
strand orientation, missing/edge policy, размерность и ключ агрегации.


In [7]:
from ml_project.epic_experiments.exp_010_regionalnye_dinukleotidy_v_okne_201_bp import (
    FEATURE_READY, count_candidate, cross_validate_candidate,
    diagnose_feature, evaluate_candidate, feature_contract, fit_candidate,
)

display(pd.Series(feature_contract(), name="value").to_frame())


,value
champion,EXP-009/run_001
candidates_in_priority_order,"(AA, TT, TA, GG, CC, AT, GC, CA, TG, AG, CT, G..."
formula,pair_count * valid_count / (left_count * right...
window,"strand-oriented inclusive [-100,+100]"
bins,"(<0.25, [0.25,0.50), [0.50,0.75), [0.75,1.00),..."
reference_bin,"[0.75,1.00)"
candidate_columns,7
total_model_columns,4393
aggregation_key,contig x strand x 6-mer x GC-bin x CpG-bin x p...
missing_edge_policy,clipped window; missing bin if denominator is ...


## 4. Дешёвый train-only pilot

Сначала 1–2 млн позиций и один крупный `contig × strand`: число unique rows,
память, время и биологическая sanity-check. Это защита от повторения ISSUE-001.


In [77]:
RUN_FEATURE_PILOT = True
if not FEATURE_READY:
    print("Новый признак ещё не реализован — ожидаемая точка совместной работы.")
elif RUN_FEATURE_PILOT:
    pilot = diagnose_feature(split, sequences, parameters=CANDIDATE_PARAMETERS)
    display(pilot)
else:
    print("STOP: сначала проверьте feature contract.")


,pair,pilot_contig,pilot_strand,positions,joint_rows,unique_rows_per_position,totals_memory_mib,seconds
0,GT,NC_064035.1,+,2000000,544843,0.272421,7.794168,1.078


# STOP 1 — перед полным подсчётом train


In [78]:
RUN_TRAIN_COUNT = True
if RUN_TRAIN_COUNT:
    started = time.monotonic()
    train_candidate_counts = count_candidate(
        split, sequences, "train", parameters=CANDIDATE_PARAMETERS, verbose=True)
    print(f"Train counted in {time.monotonic() - started:.2f} s")
else:
    print("Train count закрыт: RUN_TRAIN_COUNT=False")


[01/18] pair=GT NC_064035.1 strand=+ positions=16,728,378 rows=971,513 elapsed=9.0s
[02/18] pair=GT NC_064035.1 strand=- positions=16,728,378 rows=969,333 elapsed=17.7s
[03/18] pair=GT NC_064036.1 strand=+ positions=15,805,189 rows=969,219 elapsed=26.1s
[04/18] pair=GT NC_064036.1 strand=- positions=15,805,189 rows=966,681 elapsed=34.6s
[05/18] pair=GT NC_064038.1 strand=+ positions=12,608,976 rows=927,673 elapsed=41.8s
[06/18] pair=GT NC_064038.1 strand=- positions=12,608,976 rows=928,946 elapsed=49.0s
[07/18] pair=GT NC_064039.1 strand=+ positions=13,843,298 rows=943,154 elapsed=56.3s
[08/18] pair=GT NC_064039.1 strand=- positions=13,843,298 rows=943,044 elapsed=63.6s
[09/18] pair=GT NC_064040.1 strand=+ positions=13,025,510 rows=929,900 elapsed=70.8s
[10/18] pair=GT NC_064040.1 strand=- positions=13,025,510 rows=929,716 elapsed=77.9s
[11/18] pair=GT NC_064043.1 strand=+ positions=12,019,919 rows=922,999 elapsed=84.3s
[12/18] pair=GT NC_064043.1 strand=- positions=12,019,919 rows=919

# STOP 2 — перед paired train-only CV


In [79]:
RUN_CV = True
if RUN_CV:
    cv_results = cross_validate_candidate(
        train_candidate_counts, fold_assignment, champion_dir=CHAMPION_DIR,
        parameters=CANDIDATE_PARAMETERS, verbose=True)
    display(cv_results)
else:
    print("CV закрыт: RUN_CV=False")


[1/3] pair=GT fold=0 C=0.0001 AP=0.002997583 time=13.53s converged=True
[2/3] pair=GT fold=1 C=0.0001 AP=0.002498222 time=14.22s converged=True
[3/3] pair=GT fold=2 C=0.0001 AP=0.002614644 time=14.36s converged=True


,pair,fold,validation_contigs,segment,variant,average_precision,epic_spearman,positions,positives,prevalence,C,iterations,max_iter,converged,fit_seconds,aggregate_rows,features,represented_positions,warnings
0,GT,0,"NC_064035.1,NC_064043.1,NC_064048.1","NC_064035.1,NC_064043.1,NC_064048.1",EXP-010 EXP-009 + GT O/E201,0.002998,0.101247,75423552,61433,0.000815,0.0001,235,2000,True,13.531,1389167,4393,152640608,
1,GT,1,"NC_064036.1,NC_064038.1,NC_064045.1","NC_064036.1,NC_064038.1,NC_064045.1",EXP-010 EXP-009 + GT O/E201,0.002498,0.098458,77195444,56300,0.000729,0.0001,227,2000,True,14.219,1390888,4393,150868716,
2,GT,2,"NC_064039.1,NC_064040.1,NC_064047.1","NC_064039.1,NC_064040.1,NC_064047.1",EXP-010 EXP-009 + GT O/E201,0.002615,0.101396,75445164,52710,0.000699,0.0001,234,2000,True,14.359,1396039,4393,152618996,


In [80]:
if "SCREEN_CV_RESULTS" not in globals():
    SCREEN_CV_RESULTS = {}

SCREEN_CV_RESULTS[PAIR_TO_RUN] = cv_results.copy()

print("Сохранены пары:", list(SCREEN_CV_RESULTS))

Сохранены пары: ['AA', 'TT', 'AT', 'CC', 'GG', 'GC', 'CA', 'AG', 'CT', 'GA', 'AC', 'GT']


## 5. Выбор параметров и paired CV gate


In [ ]:
CV_GATE_PASSED = False  # TODO: вычислять из cv_results.
SELECTED_PARAMETERS = {}       # TODO: зафиксировать train-only выбор.
print("CV gate:", CV_GATE_PASSED)


# STOP 3 — перед full-train fit


In [ ]:
RUN_FULL_FIT = False
if RUN_FULL_FIT:
    if not CV_GATE_PASSED:
        raise RuntimeError("Candidate не прошёл paired CV gate")
    final_fit = fit_candidate(train_candidate_counts,
        selected_parameters=SELECTED_PARAMETERS,
        parameters=CANDIDATE_PARAMETERS, verbose=True)
else:
    print("Full fit закрыт: RUN_FULL_FIT=False")


# STOP 4 — перед одноразовым validation


In [ ]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    validation_counts = count_candidate(
        split, sequences, "validation", parameters=CANDIDATE_PARAMETERS, verbose=True)
    candidate_evaluation = evaluate_candidate(
        validation_counts, final_fit, parameters=CANDIDATE_PARAMETERS)
else:
    print("Validation закрыт: RUN_VALIDATION=False")


## 6. Метрики, графики и интерпретация


In [ ]:
METRICS = {}       # reference/candidate/delta AP и Spearman.
RESULT_TABLES = {}  # "name.csv": DataFrame
PLOT_OBJECTS = {}   # "name.png": Figure
INTERPRETATION = ""
DECISION = "pending"  # adopt/reject/iterate/inconclusive
print("Decision:", DECISION)


## 7. Сохранение проведённого эксперимента


In [ ]:
SAVE_RUN = False
if SAVE_RUN:
    if DECISION == "pending" or not INTERPRETATION.strip() or not METRICS:
        raise ValueError("Заполните METRICS, INTERPRETATION и DECISION")
    if (ARTIFACT_DIR / "metadata.json").exists():
        raise FileExistsError(f"Immutable run exists: {ARTIFACT_DIR}")
    record = build_run_record(SPEC, split, run_name=RUN_NAME,
        notebook_path=NOTEBOOK_PATH,
        parameters=CANDIDATE_PARAMETERS | SELECTED_PARAMETERS,
        metrics=METRICS, interpretation=INTERPRETATION, decision=DECISION)
    output = save_run_record(PROJECT_ROOT, record)
    for filename, table in RESULT_TABLES.items():
        table.to_csv(output / filename, index=False)
    plot_dir = output / "plots"
    plot_dir.mkdir(exist_ok=True)
    for filename, figure in PLOT_OBJECTS.items():
        figure.savefig(plot_dir / filename, dpi=170, bbox_inches="tight", facecolor="white")
    print("Saved run:", output)
else:
    print("SAVE_RUN=False: nothing was written.")


## 8. Отчёт в Obsidian

После сохранения добавляем report builder, который читает immutable artifacts
и обновляет только автоматический блок карточки. Ручные выводы сохраняются.
